# Evaluating the Work Clustering

*Part 1 of 3 in our evaluation of the [What Is Seattle Reading?](https://melaniewalsh.github.io/whats-seattle-reading-viewer/) viewer.*

When you open an author in the viewer and click "✨ Show top work groups," an algorithm quietly makes a lot of decisions for you. It decides that *"Harry Potter and the Sorcerer's Stone: Harry Potter Series, Book 1 (Unabridged)"* and *"Harry Potter and the sorcerer's stone / by J. K. Rowling ; illustrations by Mary Grandpre."* are the same book — one *work*, in library-speak — and it groups them into a single line on the chart.

But how often is the algorithm *right*?

Right now, the honest answer is: we don't really know. The project README says it handles "~85% of cases," but that number is a guess based on eyeballing a few authors. We're submitting this project to an academic conference ([JCDL 2026](https://2026.jcdl.org/)), and reviewers will — rightly! — want a real number with a real method behind it. In this notebook, we're going to produce that number.

By the end of this notebook, you will be able to:

- Explain what a **gold standard** is and why evaluation needs one
- Turn a clustering problem into a **pairwise classification** problem
- Compute **precision, recall, and F1** with scikit-learn — and explain what they mean in plain English
- Perform an **error analysis** that tells us *how* the algorithm fails, not just how often

## The Key Idea: We Already Have a Gold Standard

To evaluate an algorithm, you need something to compare it against — a set of answers that you trust, typically produced by careful humans. This is called a **gold standard** or **ground truth**.

Creating a gold standard is usually the most expensive part of any evaluation. But we're in luck: it already exists. For [a previous research paper](https://doi.org/10.63744/P6qPH135jhY2), we studied the 93 authors in the post-1945 volume of *The Norton Anthology of American Literature* (the NAAL — the big brick of a book that defines the American literary canon for many college classrooms). To do that study, we had to solve *exactly this problem*: we manually clustered every messy catalog title into works, by hand, recording their decisions in a column called `Normalized.Title`.

⚠️ **Heads up!** A gold standard is only "gold" if we understand how it was made. The NAAL data was clustered by a hybrid computational + manual process, and the humans made judgment calls (Is a Spanish translation of *Beloved* the same work? They said yes). As you work, stay alert for places where the gold standard itself might be inconsistent — finding those is a *contribution*, not a nuisance. Flag them; don't silently work around them.

## Setup

You'll need `pandas` and `scikit-learn`. If you don't have them:

```
pip install pandas scikit-learn
```

This notebook lives in `paper/eval/` of the [whats-seattle-reading-viewer](https://github.com/melaniewalsh/whats-seattle-reading-viewer) repository, and it imports the clustering algorithm from `normalize_work_key.py` in the same folder.

In [ ]:
import pandas as pd
from itertools import combinations
from sklearn.metrics import precision_recall_fscore_support

pd.options.display.max_colwidth = 100

## Meet the Algorithm

The viewer's clustering algorithm is a JavaScript function called `normalizeWorkKey()`. It boils every catalog title down to a normalized "work key" by stripping predictable cataloging boilerplate — the `/ by Author...` credits, the `(Unabridged)` tags, the `: Series, Book 1` subtitles. Titles that end up with the same key are treated as the same work. (The full algorithm, and the reasoning behind every rule, is documented in [`docs/title-clustering.md`](../../docs/title-clustering.md) — it's worth a read.)

We've ported that function to Python, line by line, in `normalize_work_key.py`. Let's import it and kick the tires.

In [ ]:
from normalize_work_key import normalize_work_key

titles = [
    "Harry Potter and the Sorcerer's Stone: Harry Potter Series, Book 1 (Unabridged)",
    "Harry Potter and the sorcerer's stone / by J. K. Rowling ; illustrations by Mary Grandpre.",
    "Fantastic Beasts: The Crimes of Grindelwald: The Original Screenplay",
    "Slaughterhouse-Five: or, The Children's Crusade",
]

for title in titles:
    print(f"{normalize_work_key(title)!r:45}  <-  {title}")

Notice that both *Sorcerer's Stone* variants collapse to the same key — that's a correct merge — while *The Crimes of Grindelwald* keeps its distinguishing subtitle and stays separate from other *Fantastic Beasts* titles. That's the algorithm working as designed.

⚠️ **Heads up!** As you work through this notebook, you may spot things you'd do differently in `normalize_work_key.py`. Resist the urge to "improve" it! We are evaluating **what the viewer actually does**, so the port has to stay faithful to the JavaScript. If you find what you think is a bug, write it down and tell us — that's genuinely valuable — but don't fix it silently, or we'll end up publishing numbers for an algorithm that doesn't exist.

## Loading the Gold Standard

The gold-standard data is a CSV of Seattle Public Library checkouts for all 93 NAAL authors, 2005–2025. Every row is one month of checkouts for one catalog title in one format. The two columns we care most about:

- `Title` — the **raw, messy** catalog title (this is what our algorithm sees)
- `Normalized.Title` — the **human-assigned work** (this is the right answer)

And similarly `Creator` (raw) and `Normalized.Creator` (the human-normalized author). The file is publicly hosted, so we can load it straight from the URL — it may take a minute.

In [ ]:
GOLD_URL = (
    "https://seattle-library-checkout-data.s3.us-west-2.amazonaws.com/"
    "norton-anthology_spl-checkouts_2005-2025.csv"
)

gold = pd.read_csv(GOLD_URL, low_memory=False)  # low_memory=False: the ISBN column mixes types
print(f"{len(gold):,} rows")
gold.head(3)

In [ ]:
gold.columns.tolist()

## From Checkout Rows to Unique Titles

Remember: this data has one row per *month* per *format*. The same catalog title appears hundreds of times (once for October 2016, once for November 2016, ...). But clustering operates on **unique title strings**, so our first move is to deduplicate.

We'll also filter to books — print books, ebooks, and audiobooks — to match what the viewer and the original study analyze. Let's look at what `MaterialType` contains first, so the filter isn't a leap of faith:

In [ ]:
gold["MaterialType"].value_counts()

In [ ]:
BOOK_TYPES = ["BOOK", "EBOOK", "AUDIOBOOK"]

titles_df = (
    gold[gold["MaterialType"].isin(BOOK_TYPES)]
    [["Normalized.Creator", "Title", "Normalized.Title"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

print(f"{len(titles_df):,} unique (author, title) rows")
print(f"{titles_df['Normalized.Creator'].nunique()} authors")
titles_df.sample(5, random_state=42)

💡 **Tip:** Take a minute to browse this dataframe. Pick an author you know — Toni Morrison, say — and look at how many raw `Title` variants map onto one `Normalized.Title`. This is the mess the algorithm has to clean up, and you'll evaluate it better if you've seen the mess up close.

In [ ]:
# Your turn: look at one author's titles. How messy are they?
titles_df[titles_df["Normalized.Creator"] == "Toni Morrison"].sort_values("Normalized.Title").head(15)

## The Trick: Clustering → Pairs

Here's the conceptual heart of this notebook, so let's slow down.

How do you grade a *clustering*? The algorithm's clusters and the humans' clusters might carve up the titles differently — different numbers of groups, different sizes. You can't just ask "what percent of clusters are right?" because there's no one-to-one correspondence between machine clusters and human clusters.

The standard solution is lovely: **stop looking at clusters and start looking at pairs.** For every possible *pair* of titles by the same author, ask two yes/no questions:

1. Did the **humans** put these two titles in the same work?
2. Did the **algorithm** put these two titles in the same work?

Now clustering has become a binary classification problem — something we know exactly how to grade. Every pair lands in one of four boxes:

| | Algorithm says: same work | Algorithm says: different works |
|---|---|---|
| **Humans say: same work** | ✅ true positive | ❌ **false split** (missed merge) |
| **Humans say: different works** | ❌ **false merge** | ✅ true negative |

And from those boxes come the two numbers that matter:

- **Precision** — *when the algorithm merges two titles, how often is it right?* Low precision means **false merges**: different books getting mushed into one line on the chart.
- **Recall** — *of the merges the humans made, how many did the algorithm find?* Low recall means **false splits**: one book scattered across several lines.

**F1** is the harmonic mean of the two — a single summary number that punishes you for sacrificing either one.

💡 **Tip:** Precision and recall failures *feel* different to a user. A false split is visible and annoying (why are there two *Beloved* lines?). A false merge is invisible and misleading (the chart quietly combines two different books). Keep that asymmetry in mind when you interpret your results.

## A Toy Example First

Before running on 93 authors, let's make the machinery transparent with four titles we can grade in our heads. Suppose the humans say A and B are the same work, and C and D are each their own work:

In [ ]:
# (title, human-assigned work id)
toy = [
    ("Beloved",                          "beloved"),
    ("Beloved: A Novel (Unabridged)",    "beloved"),
    ("Song of Solomon",                  "song-of-solomon"),
    ("Sula",                             "sula"),
]

y_true, y_pred = [], []
for (title1, work1), (title2, work2) in combinations(toy, 2):
    human_says_same = int(work1 == work2)
    key1, key2 = normalize_work_key(title1), normalize_work_key(title2)
    algo_says_same = int(key1 == key2 and key1 != "")
    y_true.append(human_says_same)
    y_pred.append(algo_says_same)
    print(f"human={human_says_same}  algo={algo_says_same}   {title1!r} vs {title2!r}")

Four titles make six pairs (`combinations` gives us every unordered pair). Only one pair should merge — the two *Beloveds* — and you can check by eye whether the algorithm agreed.

Now let's compute precision and recall two ways: by hand from the definitions, and with scikit-learn. They should match exactly — if they ever don't, something is wrong with how we built the pairs, and we want to know *before* we trust the big run.

In [ ]:
# By hand, straight from the definitions
true_positives  = sum(1 for t, p in zip(y_true, y_pred) if t == 1 and p == 1)
false_merges    = sum(1 for t, p in zip(y_true, y_pred) if t == 0 and p == 1)
false_splits    = sum(1 for t, p in zip(y_true, y_pred) if t == 1 and p == 0)

precision_by_hand = true_positives / (true_positives + false_merges)
recall_by_hand    = true_positives / (true_positives + false_splits)
print(f"by hand:  precision={precision_by_hand:.3f}  recall={recall_by_hand:.3f}")

# With scikit-learn
p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="binary", zero_division=0)
print(f"sklearn:  precision={p:.3f}  recall={r:.3f}  f1={f1:.3f}")

⚠️ **Heads up!** Why `average="binary"`? Because we only want scores for the *positive* class (pairs that should merge). Most pairs of titles are different works — that's the boring, easy majority. If we scored both classes, or reported plain accuracy, the algorithm could look brilliant by just never merging anything. This is the classic **class imbalance** problem, and scoring only the positive class is our defense against it.

One more subtlety hiding in the toy code: the `key1 != ""` check. If `normalize_work_key()` strips a title down to nothing (it can happen with unusual titles), we must **not** let all those empty-key titles merge with each other — an empty key means "I don't know," not "these match." 

## The Real Thing: All 93 Authors

Now we scale the exact same logic up. Two rules to respect:

1. **Compare within authors only.** The viewer only ever clusters one author's titles at a time, so pairs are built per author — we never ask whether a Morrison title matches a Baldwin title.
2. **Micro-average.** We pool every pair from every author into one big list before scoring, so authors with many titles count for more — which mirrors how much of the data they actually are.

In [ ]:
y_true, y_pred = [], []
per_author_rows = []

for author, group in titles_df.groupby("Normalized.Creator"):
    rows = list(group[["Title", "Normalized.Title"]].itertuples(index=False))
    if len(rows) < 2:
        continue  # can't make a pair out of one title

    author_true, author_pred = [], []
    for (title1, work1), (title2, work2) in combinations(rows, 2):
        key1, key2 = normalize_work_key(title1), normalize_work_key(title2)
        author_true.append(int(work1 == work2))
        author_pred.append(int(key1 == key2 and key1 != ""))

    y_true.extend(author_true)
    y_pred.extend(author_pred)

    p, r, f1, _ = precision_recall_fscore_support(
        author_true, author_pred, average="binary", zero_division=0
    )
    per_author_rows.append(
        {"author": author, "n_titles": len(rows), "n_pairs": len(author_true),
         "precision": p, "recall": r, "f1": f1}
    )

p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="binary", zero_division=0)
print(f"OVERALL (micro-averaged over {len(y_true):,} pairs)")
print(f"precision={p:.3f}  recall={r:.3f}  f1={f1:.3f}")

🎉 There it is — the number that replaces "~85%, probably." Write it down, but don't stop here: a single overall score hides *where* the algorithm struggles, and that's what the next two sections dig into.

In [ ]:
per_author = pd.DataFrame(per_author_rows).sort_values("f1")

# The five authors the algorithm finds hardest
per_author.head(5)

In [ ]:
# And the five it finds easiest
per_author.tail(5)

## Error Analysis: *How* Does It Fail?

For the paper — and for improving the viewer someday — the failure *examples* matter as much as the scores. Let's pull out the actual disagreements: **false merges** (algorithm merged, humans didn't — these hurt precision) and **false splits** (humans merged, algorithm didn't — these hurt recall).

In [ ]:
false_merges, false_splits = [], []

for author, group in titles_df.groupby("Normalized.Creator"):
    rows = list(group[["Title", "Normalized.Title"]].itertuples(index=False))
    for (title1, work1), (title2, work2) in combinations(rows, 2):
        key1, key2 = normalize_work_key(title1), normalize_work_key(title2)
        human = work1 == work2
        algo = key1 == key2 and key1 != ""
        if algo and not human:
            false_merges.append((author, title1, title2))
        elif human and not algo:
            false_splits.append((author, title1, title2))

print(f"{len(false_merges):,} false-merge pairs, {len(false_splits):,} false-split pairs")

In [ ]:
# Look at a sample of each. What patterns do you see?
print("=== FALSE MERGES (algorithm merged; humans kept separate) ===")
for author, t1, t2 in false_merges[:10]:
    print(f"  [{author}]")
    print(f"     {t1!r}")
    print(f"     {t2!r}")

print()
print("=== FALSE SPLITS (humans merged; algorithm kept separate) ===")
for author, t1, t2 in false_splits[:10]:
    print(f"  [{author}]")
    print(f"     {t1!r}")
    print(f"     {t2!r}")

## Your Turn: Reading the Errors Like a Humanist

This is where your judgment matters more than scikit-learn's. Go through a good sample of the errors above (not just the first 10 — try slicing into the middle) and sort them into piles. Questions to guide you:

1. **Are the false splits systematic?** Translations, for example: the humans clustered *Billŏbidŭ* (the Korean *Beloved*) with *Beloved*; no boilerplate-stripping rule could ever do that. How many false splits are translations? These may be worth reporting as their own category — "the algorithm doesn't attempt translations" is a different (and more forgivable) claim than "the algorithm misses merges."
2. **Are any errors actually gold-standard errors?** Sometimes you'll disagree with the human clusterers. If a pair looks like the *humans* got it wrong, note it — a benchmark's imperfections belong in the paper too.
3. **Is a whole category of title driving the failures?** Collections, anthologies, "The Complete Stories of..." — short-story authors may cluster very differently from novelists. Does that explain the hardest-authors table?
4. **Pick 2–3 vivid examples** for the paper — one characteristic false merge, one characteristic false split. Concrete examples are what make an evaluation section persuasive.

## What to Upload to Gogle Dive

- [ ] This notebook, run top to bottom
- [ ] The headline numbers: overall pairwise precision, recall, F1, and the number of pairs
- [ ] The per-author table (save it: `per_author.to_csv("work_clustering_per_author.csv", index=False)`)
- [ ] A short written summary (a few paragraphs is plenty) of your error analysis: the piles you sorted errors into, rough counts, and your 2–3 vivid examples
- [ ] Any suspected bugs in the port or inconsistencies in the gold standard, flagged loudly

☕ Then take a break — you've just done real science. When you're ready, Part 2 (auditing the *author* clustering) is next.